# Network Programming Demo
### TCP · UDP · TLS — Interactive Examples

Each section below lets you:
1. **Start** a server in a background thread (non-blocking)
2. **Send** messages from a client
3. **View** the server log at any time
4. **Stop** the server cleanly

Ports used: **TCP → 9901 · UDP → 9902 · TLS → 9903**  
TLS certificates are auto-generated in `./tls_certs/`.

## 🔧 Setup
Run this cell first — it imports everything and defines the `ServerLog` helper.

In [1]:
import socket
import ssl
import threading
import time
import subprocess
import os
from collections import deque
from datetime import datetime


class ServerLog:
    """Thread-safe log buffer that prints entries immediately and stores them for later review."""

    def __init__(self, name):
        self.name = name
        self._log = deque(maxlen=200)
        self._lock = threading.Lock()

    def add(self, msg):
        ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        entry = f"[{ts}] {msg}"
        with self._lock:
            self._log.append(entry)
        print(f"[{self.name}] {entry}")

    def show(self):
        print(f"\n{'='*50}")
        print(f"  {self.name} — full log")
        print(f"{'='*50}")
        with self._lock:
            if not self._log:
                print("  (no entries yet)")
            for line in self._log:
                print(f"  {line}")
        print(f"{'='*50}\n")

    def clear(self):
        with self._lock:
            self._log.clear()


print("✅ Setup complete — ready to run examples!")

✅ Setup complete — ready to run examples!


---
## 1️⃣  TCP (Transmission Control Protocol)
Connection-oriented, reliable, ordered delivery.  
The server echoes back every message it receives.

### Start TCP Server

In [ ]:
TCP_PORT = 9901
tcp_log = ServerLog("TCP Server")
tcp_stop = threading.Event()


def handle_tcp_client(conn, addr, log):
    try:
        data = conn.recv(1024)
        if data:
            log.add(f"← Received from {addr}: {data.decode('utf-8')!r}")
            conn.sendall(data)
            log.add(f"→ Echoed back to {addr}")
    except Exception as e:
        log.add(f"Error handling {addr}: {e}")
    finally:
        conn.close()


def run_tcp_server(stop_event, log, port):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    sock.bind(("0.0.0.0", port))
    sock.listen(10)
    sock.settimeout(1.0)
    log.add(f"Listening on 0.0.0.0:{port}")
    while not stop_event.is_set():
        try:
            conn, addr = sock.accept()
            log.add(f"New connection from {addr}")
            t = threading.Thread(target=handle_tcp_client, args=(conn, addr, log), daemon=True)
            t.start()
        except socket.timeout:
            continue
        except OSError:
            break
    sock.close()
    log.add("Server stopped.")


tcp_stop.clear()
tcp_log.clear()
_t = threading.Thread(target=run_tcp_server, args=(tcp_stop, tcp_log, TCP_PORT), daemon=True)
_t.start()
time.sleep(0.2)
print(f"✅ TCP Server started on port {TCP_PORT}")

### TCP Client
Change `message` and re-run this cell as many times as you like.

In [ ]:
message = "Hello, TCP Server!"  # ← edit me

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect(("127.0.0.1", TCP_PORT))
    print(f"→ Sending:       {message!r}")
    s.sendall(message.encode("utf-8"))
    response = s.recv(1024)
    print(f"← Echo received: {response.decode('utf-8')!r}")

### View TCP Server Log

In [ ]:
tcp_log.show()

### Stop TCP Server

In [ ]:
tcp_stop.set()
time.sleep(0.4)
print("🛑 TCP Server stopped")

---
## 2️⃣  UDP (User Datagram Protocol)
Connectionless, lightweight, no guaranteed delivery.  
The server also echoes messages back. A broadcast variant is included.

### Start UDP Server

In [ ]:
UDP_PORT = 9902
udp_log = ServerLog("UDP Server")
udp_stop = threading.Event()


def run_udp_server(stop_event, log, port):
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    sock.bind(("0.0.0.0", port))
    sock.settimeout(1.0)
    log.add(f"Listening on 0.0.0.0:{port}")
    while not stop_event.is_set():
        try:
            data, addr = sock.recvfrom(1024)
            log.add(f"← Received from {addr}: {data.decode('utf-8')!r}")
            sock.sendto(data, addr)
            log.add(f"→ Echoed back to {addr}")
        except socket.timeout:
            continue
        except OSError:
            break
    sock.close()
    log.add("Server stopped.")


udp_stop.clear()
udp_log.clear()
_t = threading.Thread(target=run_udp_server, args=(udp_stop, udp_log, UDP_PORT), daemon=True)
_t.start()
time.sleep(0.2)
print(f"✅ UDP Server started on port {UDP_PORT}")

### UDP Client
Sends a single datagram and waits for the echo.

In [ ]:
message = "Hello, UDP Server!"  # ← edit me

with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
    s.settimeout(2.0)
    print(f"→ Sending:       {message!r}")
    s.sendto(message.encode("utf-8"), ("127.0.0.1", UDP_PORT))
    data, addr = s.recvfrom(1024)
    print(f"← Echo received from {addr}: {data.decode('utf-8')!r}")

### UDP Broadcast Client
Sends to the broadcast address `255.255.255.255` — any host on the LAN listening on the same port
could receive it.  On most OSes, localhost will loop the broadcast back through the server above.

In [ ]:
broadcast_msg = "Hello, everyone on the LAN!"  # ← edit me

with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
    s.setsockopt(socket.SOL_SOCKET, socket.SO_BROADCAST, 1)
    s.settimeout(2.0)
    print(f"→ Broadcasting:  {broadcast_msg!r}")
    s.sendto(broadcast_msg.encode("utf-8"), ("255.255.255.255", UDP_PORT))
    try:
        data, addr = s.recvfrom(1024)
        print(f"← Echo received from {addr}: {data.decode('utf-8')!r}")
    except socket.timeout:
        print("⚠️  No response (broadcast may not loop back on this OS — that is normal)")

### View UDP Server Log

In [ ]:
udp_log.show()

### Stop UDP Server

In [ ]:
udp_stop.set()
time.sleep(0.4)
print("🛑 UDP Server stopped")

---
## 3️⃣  TLS (Transport Layer Security)
TCP with encryption and certificate-based authentication.

**Steps:**
1. Generate certificates (CA + server) — run once
2. Start the TLS server
3. Connect the TLS client

### Generate TLS Certificates
Runs `openssl` to create a local CA and a server certificate signed by that CA.  
Output is written to `./tls_certs/`. Safe to re-run — it will overwrite existing certs.

In [2]:
CERT_DIR = os.path.join(os.getcwd(), "tls_certs")
os.makedirs(CERT_DIR, exist_ok=True)

# Write openssl.cnf with SAN for 127.0.0.1 and hostname "server"
openssl_cnf_content = """\
[ req ]
distinguished_name = req_distinguished_name
prompt             = no
default_md         = sha256

[ req_distinguished_name ]
C  = US
ST = MA
L  = Lowell
O  = UMass Lowell
OU = Network_Python
CN = My Test CA

[ v3_ca ]
subjectKeyIdentifier   = hash
authorityKeyIdentifier = keyid:always,issuer
basicConstraints       = critical, CA:TRUE
keyUsage               = critical, keyCertSign, cRLSign

[ server_cert ]
basicConstraints    = CA:FALSE
keyUsage            = digitalSignature, keyEncipherment
extendedKeyUsage    = serverAuth
subjectAltName      = @alt_names

[ alt_names ]
DNS.1 = server
IP.1  = 127.0.0.1
"""

CNF  = os.path.join(CERT_DIR, "openssl.cnf")
CA_KEY  = os.path.join(CERT_DIR, "ca.key")
CA_CRT  = os.path.join(CERT_DIR, "ca.crt")
SRV_KEY = os.path.join(CERT_DIR, "server.key")
SRV_CSR = os.path.join(CERT_DIR, "server.csr")
SRV_PEM = os.path.join(CERT_DIR, "server.pem")

with open(CNF, "w") as f:
    f.write(openssl_cnf_content)


def run_openssl(args, desc):
    result = subprocess.run(["openssl"] + args, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✅ {desc}")
    else:
        print(f"  ❌ {desc}")
        print(f"     {result.stderr.strip()}")
    return result.returncode == 0


print(f"🔑 Generating TLS certificates in: {CERT_DIR}\n")

steps = [
    (["genpkey", "-algorithm", "RSA", "-out", CA_KEY],
     "CA private key"),

    (["req", "-x509", "-new", "-key", CA_KEY, "-sha256",
      "-days", "3650", "-out", CA_CRT,
      "-subj", "/CN=My Test CA",
      "-config", CNF, "-extensions", "v3_ca"],
     "CA self-signed certificate"),

    (["genpkey", "-algorithm", "RSA", "-out", SRV_KEY],
     "Server private key"),

    (["req", "-new", "-key", SRV_KEY, "-out", SRV_CSR,
      "-subj", "/CN=server"],
     "Server certificate signing request (CSR)"),

    (["x509", "-req", "-in", SRV_CSR,
      "-CA", CA_CRT, "-CAkey", CA_KEY, "-CAserial", os.path.join(CERT_DIR, "ca.srl"), "-CAcreateserial",
      "-out", SRV_PEM, "-days", "825", "-sha256",
      "-extfile", CNF, "-extensions", "server_cert"],
     "Server certificate (signed by CA)"),
]

ok = all(run_openssl(args, desc) for args, desc in steps)

if ok:
    print(f"\n✅ All certificates generated successfully!")
    print(f"   CA cert:     {CA_CRT}")
    print(f"   Server cert: {SRV_PEM}")
    print(f"   Server key:  {SRV_KEY}")
else:
    print("\n❌ Certificate generation failed — check errors above.")

🔑 Generating TLS certificates in: /Users/justi/LOCAL_CODE/COMP.2300/Network_Python/tls_certs

  ✅ CA private key
  ✅ CA self-signed certificate
  ✅ Server private key
  ✅ Server certificate signing request (CSR)
  ✅ Server certificate (signed by CA)

✅ All certificates generated successfully!
   CA cert:     /Users/justi/LOCAL_CODE/COMP.2300/Network_Python/tls_certs/ca.crt
   Server cert: /Users/justi/LOCAL_CODE/COMP.2300/Network_Python/tls_certs/server.pem
   Server key:  /Users/justi/LOCAL_CODE/COMP.2300/Network_Python/tls_certs/server.key


### Start TLS Server
The server logs the negotiated cipher suite and protocol version for each connection.

In [3]:
TLS_PORT = 9903
tls_log = ServerLog("TLS Server")
tls_stop = threading.Event()


def load_tls_server_context():
    ctx = ssl.create_default_context(ssl.Purpose.CLIENT_AUTH)
    ctx.load_cert_chain(certfile=SRV_PEM, keyfile=SRV_KEY)
    ctx.verify_mode = ssl.CERT_OPTIONAL   # clients need not present a cert
    ctx.load_verify_locations(cafile=CA_CRT)
    return ctx


def handle_tls_client(tls_conn, addr, log):
    try:
        cipher, proto, bits = tls_conn.cipher()
        log.add(f"🔒 TLS handshake OK with {addr} — {proto}, cipher={cipher}, bits={bits}")
        data = tls_conn.recv(1024)
        if data:
            log.add(f"← Received: {data.decode('utf-8')!r}")
            tls_conn.sendall(data)
            log.add(f"→ Echoed back")
    except Exception as e:
        log.add(f"Error: {e}")
    finally:
        tls_conn.close()


def run_tls_server(stop_event, log, port):
    ssl_ctx = load_tls_server_context()
    raw_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    raw_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    raw_sock.bind(("0.0.0.0", port))
    raw_sock.listen(10)
    raw_sock.settimeout(1.0)
    log.add(f"Listening on 0.0.0.0:{port} (TLS)")
    while not stop_event.is_set():
        try:
            conn, addr = raw_sock.accept()
            log.add(f"New TCP connection from {addr} — performing TLS handshake…")
            tls_conn = ssl_ctx.wrap_socket(conn, server_side=True)
            t = threading.Thread(target=handle_tls_client, args=(tls_conn, addr, log), daemon=True)
            t.start()
        except socket.timeout:
            continue
        except ssl.SSLError as e:
            log.add(f"TLS error: {e}")
        except OSError:
            break
    raw_sock.close()
    log.add("Server stopped.")


tls_stop.clear()
tls_log.clear()
_t = threading.Thread(target=run_tls_server, args=(tls_stop, tls_log, TLS_PORT), daemon=True)
_t.start()
time.sleep(0.2)
print(f"✅ TLS Server started on port {TLS_PORT}")

[TLS Server] [16:13:47.097] Listening on 0.0.0.0:9903 (TLS)
✅ TLS Server started on port 9903


[TLS Server] [16:14:19.861] New TCP connection from ('127.0.0.1', 65064) — performing TLS handshake…
[TLS Server] [16:14:19.866] 🔒 TLS handshake OK with ('127.0.0.1', 65064) — TLSv1/SSLv3, cipher=ECDHE-RSA-CHACHA20-POLY1305, bits=256
[TLS Server] [16:14:19.867] ← Received: 'Hello, TLS Server!'
[TLS Server] [16:14:19.867] → Echoed back


### TLS Client
The client verifies the server's certificate against our local CA.  
Notice the cipher suite and protocol printed below — this is the encrypted channel.

In [4]:
message = "Hello, TLS Server!"  # ← edit me

ctx = ssl.create_default_context(ssl.Purpose.SERVER_AUTH, cafile=CA_CRT)

raw_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
raw_sock.connect(("127.0.0.1", TLS_PORT))
tls_sock = ctx.wrap_socket(raw_sock, server_hostname="server")

try:
    cipher, proto, bits = tls_sock.cipher()
    print(f"🔒 Connected — {proto}, cipher={cipher}, bits={bits}")
    print(f"   Server cert CN: {tls_sock.getpeercert()['subject'][0][0][1]}")
    print()
    print(f"→ Sending:       {message!r}")
    tls_sock.sendall(message.encode("utf-8"))
    response = tls_sock.recv(1024)
    print(f"← Echo received: {response.decode('utf-8')!r}")
finally:
    tls_sock.close()

🔒 Connected — TLSv1/SSLv3, cipher=ECDHE-RSA-CHACHA20-POLY1305, bits=256
   Server cert CN: server

→ Sending:       'Hello, TLS Server!'
← Echo received: 'Hello, TLS Server!'


### View TLS Server Log

In [ ]:
tls_log.show()

### Stop TLS Server

In [ ]:
tls_stop.set()
time.sleep(0.4)
print("🛑 TLS Server stopped")

---
## Summary

| Protocol | Port | Key traits |
|----------|------|------------|
| TCP | 9901 | Connection-oriented, reliable, ordered |
| UDP | 9902 | Connectionless, fast, no delivery guarantee |
| TLS | 9903 | TCP + encryption + certificate authentication |

### Quick reference — stop all servers at once

In [ ]:
for event, name in [(tcp_stop, "TCP"), (udp_stop, "UDP"), (tls_stop, "TLS")]:
    event.set()
time.sleep(0.5)
print("🛑 All servers stopped")